# TẢI ZIP TN1 · TN2 · TN3 — gộp về layout `runs/`

Notebook **chỉ đọc**. Không train, không clone mã, không xoá gì.

Lấy mọi tệp nén của các thực nghiệm sau trên Drive, gộp các bản trùng
(kiểm khớp byte), xếp về đúng layout `runs/<thực nghiệm>/` rồi bấm tải:

| thực nghiệm | tệp nén Drive | notebook |
|---|---|---|
| `tn1` | `tn1_ds_tcn_rf61_c64` · `tn1_ds_tcn_rf61_c192` · `tn1_tcn_weightnorm` | TN1_DS_TCN_RF61_… · TN1_TCN_DSTCN_model_selection |
| `tn1_ghij` | `tn1_ghij` | TN1_final_evaluation |
| `tn2` | `tn2_ds_tcn_revin` | TN2_DS_TCN_RevIN |
| `tn2_rf` | `tn2_rf_c64_4fold` · `tn2_rf_c192_4fold` · `tn2_rf_ds_tcn_c64` | TN2_ReceptiveField_… |
| `tn3` | `tn3_ds_tcn_c64` · `tn3_ds_tcn_c64_k5` · `tn3_ds_tcn_c192` | TN3_HybridLoss_… |

TN4 dùng notebook riêng `TAI_ZIP_TN4.ipynb`.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Chọn giữ hay bỏ checkpoint

`GIU_CHECKPOINT = True` giữ `final.pth` (nặng hơn nhiều — TN3 có 10 alpha × 4 fold). `False` chỉ giữ điểm, đường cong loss, bảng lựa chọn kênh.

In [ ]:
GIU_CHECKPOINT = True

## 3. Liệt kê tệp nén

In [ ]:
import glob, os, subprocess

SRC = "/content/drive/MyDrive/mobivital"
PATS = ["tn1_*.zip", "tn2_*.zip", "tn3_*.zip"]     # bắt cả tn1_ghij, tn2_rf
BO   = {"tn1_gru_h77.zip", "tn1_bilstm_h41.zip", "tn1_cnn_lstm_h58.zip",
        "tn1_modern_tcn.zip", "tn1_mix_linear.zip", "tn1_lstm.zip",
        "tn1_ghij_lstm_h67.zip"}                   # ngoài nhánh submission

zips = sorted({p for pat in PATS for p in glob.glob(SRC + "/" + pat)
               if os.path.basename(p) not in BO and not os.path.basename(p).startswith("tn4_")})

tong = 0
print("  %-56s %8s   %s" % ("tệp", "MB", "sửa"))
print("  " + "-" * 84)
for p in zips:
    mb = os.path.getsize(p) / 1048576
    tong += mb
    g = subprocess.run(["date","-r",p,"+%m-%d %H:%M"], capture_output=True, text=True).stdout.strip()
    print("  %-56s %8.1f   %s" % (os.path.basename(p)[:56], mb, g))
print("  " + "-" * 84)
print("  %d tệp · %.0f MB" % (len(zips), tong))

## 4. Gộp về `runs/<thực nghiệm>/`

Layout đúng như `unzip <tệp>.zip -d runs/`: mỗi thực nghiệm một thư mục, trong đó `<cấu hình>_<fold>/{final.pth, curve.csv}`, `scores_*.csv`, `summary.csv` gộp khử trùng.

In [ ]:
import shutil, tarfile, tempfile, hashlib, csv

def sha(p): return hashlib.sha256(open(p,"rb").read()).hexdigest()

raw = tempfile.mkdtemp()
for p in zips:
    subprocess.run(["unzip","-oq",p,"-d",
                    os.path.join(raw, os.path.splitext(os.path.basename(p))[0])], check=True)

out_root = os.path.join(tempfile.mkdtemp(), "runs_TN123")
os.makedirs(out_root)

xung_dot = []
def chep(src, dst):
    if os.path.exists(dst):
        if sha(src) != sha(dst): xung_dot.append(dst)
        return
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)

# mọi thư mục runs/<exp>/ nằm trong <zip_name>/<exp>/
exps = set()
for d in glob.glob(raw + "/*/*"):
    if os.path.isdir(d):
        exps.add(os.path.basename(d))

n_pth = n_file = 0
for exp in sorted(exps):
    for src in glob.glob(raw + "/*/" + exp + "/**/*", recursive=True):
        if os.path.isdir(src): continue
        rel = src.split("/" + exp + "/", 1)[1]
        if rel == "summary.csv": continue                      # gộp riêng bên dưới
        if rel.endswith(".pth") and not GIU_CHECKPOINT: continue
        chep(src, os.path.join(out_root, exp, rel))
        n_file += 1
        if rel.endswith(".pth"): n_pth += 1

    # summary.csv gộp mọi dòng, khử trùng theo run_id
    rows, hdr = {}, None
    for f in glob.glob(raw + "/*/" + exp + "/summary.csv"):
        for r in csv.DictReader(open(f)):
            if hdr is None: hdr = list(r.keys())
            rows[r["run_id"]] = r
    if rows:
        with open(os.path.join(out_root, exp, "summary.csv"), "w", newline="") as fo:
            w = csv.DictWriter(fo, fieldnames=hdr); w.writeheader()
            for k in sorted(rows): w.writerow(rows[k])

archive = "/content/runs_TN123.tar.gz"
with tarfile.open(archive, "w:gz") as tf:
    tf.add(out_root, arcname="runs_TN123")

mb = os.path.getsize(archive) / 1048576
print("  thực nghiệm:", ", ".join(sorted(exps)))
print("  %d file (%d final.pth)" % (n_file, n_pth))
if xung_dot:
    print("  !! %d file trùng KHÁC byte:" % len(xung_dot))
    for x in xung_dot[:5]: print("     ", x)
else:
    print("  mọi bản trùng khớp byte")
print("  -> %s   %.1f MB" % (archive, mb))

## 5. Cây thư mục

In [ ]:
print(subprocess.run(["bash","-lc",
      "cd %s && find runs_TN123 -maxdepth 2 | sort" % os.path.dirname(out_root)],
      capture_output=True, text=True).stdout)

## 6. Tải về

In [ ]:
from google.colab import files
files.download("/content/runs_TN123.tar.gz")

## 7. Ở máy

```bash
tar -xzf runs_TN123.tar.gz
```

Ra `runs_TN123/tn1/`, `tn1_ghij/`, `tn2/`, `tn2_rf/`, `tn3/` — copy vào `runs/`
của repo. `.pth` sẽ bị `.gitignore` chặn trừ khi whitelist thêm như đã làm với
`runs/tn4/`.